# EDA Redundant Pair Compare Raw

전처리 이전(raw에 가까운) 이중화 계량기쌍 `P` 비교 노트북입니다. 기본 대상은 `H1.Z20`와 `H1.ZE20`입니다.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path('/home/playdata2/final_pj/energy-platform')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.meter_metadata import get_all_meters, get_metadata, get_redundant_pair
from scripts.preprocess_h1z16 import fetch_joined_data

candidate_pairs = []
seen = set()
for meter_urn in get_all_meters():
    pair = get_redundant_pair(meter_urn)
    if not pair:
        continue
    pair_key = tuple(sorted([meter_urn, pair]))
    if pair_key in seen:
        continue
    seen.add(pair_key)
    candidate_pairs.append({
        'main_meter': meter_urn,
        'pair_meter': pair,
        'main_desc': get_metadata(meter_urn).get('description'),
        'pair_desc': get_metadata(pair).get('description'),
    })
candidate_pairs_df = pd.DataFrame(candidate_pairs)
display(candidate_pairs_df)

MAIN_METER_URN = 'H1.Z20'
PAIR_METER_URN = get_redundant_pair(MAIN_METER_URN)
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'redundant_pair_compare_raw'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('main meter:', MAIN_METER_URN)
print('pair meter:', PAIR_METER_URN)
print('main metadata:', get_metadata(MAIN_METER_URN))
print('pair metadata:', get_metadata(PAIR_METER_URN))

## 1. Raw 데이터 로드

In [ ]:
main_df = fetch_joined_data(MAIN_METER_URN)
pair_df = fetch_joined_data(PAIR_METER_URN)

compare_df = (
    main_df[['ts', 'P']]
    .rename(columns={'P': f'P_{MAIN_METER_URN}'})
    .merge(
        pair_df[['ts', 'P']].rename(columns={'P': f'P_{PAIR_METER_URN}'}),
        on='ts',
        how='inner',
    )
    .sort_values('ts')
    .reset_index(drop=True)
)

compare_df['signed_diff'] = compare_df[f'P_{MAIN_METER_URN}'] - compare_df[f'P_{PAIR_METER_URN}']
compare_df['abs_diff'] = compare_df['signed_diff'].abs()
denom = compare_df[[f'P_{MAIN_METER_URN}', f'P_{PAIR_METER_URN}']].abs().max(axis=1)
compare_df['rel_diff_pct'] = np.where(denom > 0, compare_df['abs_diff'] / denom * 100, np.nan)

print('main rows:', len(main_df))
print('pair rows:', len(pair_df))
print('common rows:', len(compare_df))
display(compare_df.head())

## 2. Raw 기초 수치 비교

In [ ]:
corr = compare_df[[f'P_{MAIN_METER_URN}', f'P_{PAIR_METER_URN}']].corr().iloc[0, 1]
summary_df = pd.DataFrame([
    {
        'main_meter_urn': MAIN_METER_URN,
        'pair_meter_urn': PAIR_METER_URN,
        'common_rows': len(compare_df),
        'main_mean_P': compare_df[f'P_{MAIN_METER_URN}'].mean(),
        'pair_mean_P': compare_df[f'P_{PAIR_METER_URN}'].mean(),
        'mean_abs_diff': compare_df['abs_diff'].mean(),
        'median_abs_diff': compare_df['abs_diff'].median(),
        'max_abs_diff': compare_df['abs_diff'].max(),
        'mean_rel_diff_pct': compare_df['rel_diff_pct'].mean(),
        'p95_rel_diff_pct': compare_df['rel_diff_pct'].quantile(0.95),
        'corr': corr,
    }
])
display(summary_df)

top_diff_df = compare_df.sort_values('abs_diff', ascending=False)[[
    'ts',
    f'P_{MAIN_METER_URN}',
    f'P_{PAIR_METER_URN}',
    'signed_diff',
    'abs_diff',
    'rel_diff_pct',
]].head(10)
display(top_diff_df)

summary_path = OUTPUT_DIR / f'{MAIN_METER_URN.replace('.', '_')}__{PAIR_METER_URN.replace('.', '_')}_raw_summary.csv'
top_diff_path = OUTPUT_DIR / f'{MAIN_METER_URN.replace('.', '_')}__{PAIR_METER_URN.replace('.', '_')}_raw_top_diff.csv'
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
top_diff_df.to_csv(top_diff_path, index=False, encoding='utf-8-sig')
print('saved:', summary_path)
print('saved:', top_diff_path)

## 3. Raw 시계열 비교

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=False)
axes[0].plot(compare_df['ts'], compare_df[f'P_{MAIN_METER_URN}'], label=MAIN_METER_URN, linewidth=0.9)
axes[0].plot(compare_df['ts'], compare_df[f'P_{PAIR_METER_URN}'], label=PAIR_METER_URN, linewidth=0.9)
axes[0].set_title(f'{MAIN_METER_URN} vs {PAIR_METER_URN} - RAW P Timeseries')
axes[0].set_ylabel('P')
axes[0].legend()

axes[1].plot(compare_df['ts'], compare_df['signed_diff'], color='tomato', linewidth=0.8)
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Signed Difference (Raw)')
axes[1].set_ylabel('P diff')

axes[2].scatter(compare_df[f'P_{MAIN_METER_URN}'], compare_df[f'P_{PAIR_METER_URN}'], s=8, alpha=0.35, color='slateblue')
min_val = float(min(compare_df[f'P_{MAIN_METER_URN}'].min(), compare_df[f'P_{PAIR_METER_URN}'].min()))
max_val = float(max(compare_df[f'P_{MAIN_METER_URN}'].max(), compare_df[f'P_{PAIR_METER_URN}'].max()))
axes[2].plot([min_val, max_val], [min_val, max_val], color='gray', linestyle='--', linewidth=1)
axes[2].set_title('Scatter Comparison (Raw)')
axes[2].set_xlabel(MAIN_METER_URN)
axes[2].set_ylabel(PAIR_METER_URN)

plt.tight_layout()
plot_path = OUTPUT_DIR / f'{MAIN_METER_URN.replace('.', '_')}__{PAIR_METER_URN.replace('.', '_')}_raw_P_compare.png'
plt.savefig(plot_path, dpi=150)
plt.close(fig)
display(Image(filename=str(plot_path)))
print('saved:', plot_path)